In [ ]:
#!/usr/bin/env python3
# coding: utf-8

import socket
import struct
import numpy as np
from pynq import Overlay, allocate
from pynq.overlays.RVSDA import *

# ------------------------------------------------------------
# 1. Load overlay and DDR buffer
# ------------------------------------------------------------
ol = Overlay("RVSDA.bit")
rvsda = ol.RVSDA_0

data_buffer = allocate(shape=(8192,), dtype=np.uint64)
data_buffer[:] = 0
rvsda.ddr_base = data_buffer.device_address
data_buffer.flush()

# ------------------------------------------------------------
# 2. Load instructions
# ------------------------------------------------------------
instructions = np.fromfile(
    "/home/xilinx/pynq/overlays/RVSDA/firmware/firmware_mnist_net_test.bin",
    dtype=np.uint32
)
rvsda.load_instruction_memory(instructions)

# ------------------------------------------------------------
# 3. Load weights and compute offsets
# ------------------------------------------------------------
scales = np.load("/home/xilinx/pynq/overlays/RVSDA/mnist_test/scales.npz")
conv_weights = np.load("/home/xilinx/pynq/overlays/RVSDA/mnist_test/conv_weights.npy")
dense_weights = np.load("/home/xilinx/pynq/overlays/RVSDA/mnist_test/dense_weights.npy")
conv_scale = float(scales["conv_weight_scale"])
dense_scale = float(scales["dense_weight_scale"])

q_conv = quantize_symmetric(conv_weights, conv_scale)
q_dense = quantize_symmetric(dense_weights, dense_scale)

# Load a known MNIST image to test the hardware
mnist_data = np.load("/home/xilinx/pynq/overlays/RVSDA/mnist_test/mnist_test_full.npz")
test_image = np.load("/home/xilinx/pynq/overlays/RVSDA/mnist_test/test_image.npy")
test_label = np.load("/home/xilinx/pynq/overlays/RVSDA/mnist_test/test_label.npy")

# Preload weights using a dummy image (same size)
dummy_image = np.zeros((28, 28), dtype=np.float32)
activation_offset, output_offset = preload_weights(
    data_buffer,
    dummy_image,
    q_conv,
    q_dense
)

print("[Server] Offsets ready:", activation_offset, output_offset)

# ------------------------------------------------------------
# Self‑test with a known image
# ------------------------------------------------------------
print("[Server] Running self‑test on known MNIST image...")
dense_out, _ = run_network(
    rvsda,
    data_buffer,
    test_image,
    q_conv,
    q_dense,
    scales,
    a_offset=activation_offset,
    out_offset=output_offset
)
pred = np.argmax(dense_out)
print(f"[Server] Self‑test: label={test_label}, prediction={pred}, output={dense_out}")

if pred != test_label:
    print("[Server]  Self‑test failed – hardware or weights issue!")
else:
    print("[Server]  Self‑test passed – overlay works.")

# ------------------------------------------------------------
# 4. TCP server – keep connection alive for multiple frames
# ------------------------------------------------------------
HOST = '0.0.0.0'
PORT = 12345

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as server_sock:
    server_sock.bind((HOST, PORT))
    server_sock.listen(1)
    print(f"[Server] Listening on port {PORT}")

    while True:
        conn, addr = server_sock.accept()
        print(f"[Server] Connected by {addr}")
        with conn:
            while True:
                try:
                    # Receive exactly 3136 bytes (28*28 floats)
                    data = b''
                    while len(data) < 3136:
                        chunk = conn.recv(3136 - len(data))
                        if not chunk:
                            print("[Server] Client disconnected")
                            break
                        data += chunk

                    if len(data) < 3136:
                        break

                    # Convert to float32 array
                    image = np.frombuffer(data, dtype=np.float32).reshape(28, 28)

                    # ---- Debug: print received image stats ----
                    print(f"Received image: min={np.min(image):.3f}, max={np.max(image):.3f}, mean={np.mean(image):.3f}")

                    # Run inference
                    dense_out, _ = run_network(
                        rvsda,
                        data_buffer,
                        image,
                        q_conv,
                        q_dense,
                        scales,
                        a_offset=activation_offset,
                        out_offset=output_offset
                    )
#                     dense_out = run_network_sw_FP32(
#                         image,
#                         conv_weights,
#                         dense_weights
#                     )

                    pred = np.argmax(dense_out)

                    # ---- Debug: print raw output ----
#                     print(f"Dense output: {dense_out}, prediction={pred}")

                    conn.sendall(struct.pack('B', pred))
                    print(f"[Server] Sent prediction: {pred}")

                except (ConnectionResetError, BrokenPipeError):
                    print("[Server] Connection lost")
                    break
                except Exception as e:
                    print(f"[Server] Error: {e}")
                    try:
                        conn.sendall(struct.pack('B', 255))
                    except:
                        pass
                    break

Conv weights start at word offset: 0
Dense weights start at word offset: 128
Activations start word at: 21760
Output starts at : 32640
[Server] Offsets ready: 2720 4080
[Server] Running self‑test on known MNIST image...
[Server] Self‑test: label=[7], prediction=7, output=[[-15 -67 -17  16 -26 -19 -65  62  -9  17]]
[Server]  Self‑test passed – overlay works.
[Server] Listening on port 12345
[Server] Connected by ('192.168.1.118', 52250)
Received image: min=0.000, max=0.318, mean=0.021
[Server] Sent prediction: 2
Received image: min=0.000, max=0.424, mean=0.021
[Server] Sent prediction: 2
Received image: min=0.000, max=0.439, mean=0.020
[Server] Sent prediction: 2
Received image: min=0.000, max=0.424, mean=0.020
[Server] Sent prediction: 2
Received image: min=0.000, max=0.353, mean=0.020
[Server] Sent prediction: 2
Received image: min=0.000, max=0.349, mean=0.020
[Server] Sent prediction: 2
Received image: min=0.000, max=0.333, mean=0.019
[Server] Sent prediction: 2
Received image: min=0

Received image: min=0.000, max=0.373, mean=0.018
[Server] Sent prediction: 5
Received image: min=0.000, max=0.361, mean=0.017
[Server] Sent prediction: 5
Received image: min=0.000, max=0.310, mean=0.018
[Server] Sent prediction: 5
Received image: min=0.000, max=0.349, mean=0.019
[Server] Sent prediction: 5
Received image: min=0.000, max=0.365, mean=0.018
[Server] Sent prediction: 5
Received image: min=0.000, max=0.306, mean=0.019
[Server] Sent prediction: 5
Received image: min=0.000, max=0.278, mean=0.018
[Server] Sent prediction: 5
Received image: min=0.000, max=0.302, mean=0.018
[Server] Sent prediction: 5
Received image: min=0.000, max=0.325, mean=0.019
[Server] Sent prediction: 5
Received image: min=0.000, max=0.306, mean=0.019
[Server] Sent prediction: 5
Received image: min=0.000, max=0.314, mean=0.019
[Server] Sent prediction: 8
Received image: min=0.000, max=0.333, mean=0.018
[Server] Sent prediction: 1
Received image: min=0.000, max=0.341, mean=0.018
[Server] Sent prediction: 5

[Server] Sent prediction: 3
Received image: min=0.000, max=0.306, mean=0.019
[Server] Sent prediction: 3
Received image: min=0.000, max=0.322, mean=0.019
[Server] Sent prediction: 3
Received image: min=0.000, max=0.322, mean=0.019
[Server] Sent prediction: 3
Received image: min=0.000, max=0.349, mean=0.019
[Server] Sent prediction: 3
Received image: min=0.000, max=0.373, mean=0.018
[Server] Sent prediction: 3
Received image: min=0.000, max=0.396, mean=0.018
[Server] Sent prediction: 3
Received image: min=0.000, max=0.337, mean=0.019
[Server] Sent prediction: 3
Received image: min=0.000, max=0.337, mean=0.019
[Server] Sent prediction: 7
Received image: min=0.000, max=0.290, mean=0.019
[Server] Sent prediction: 3
Received image: min=0.000, max=0.325, mean=0.019
[Server] Sent prediction: 3
Received image: min=0.000, max=0.333, mean=0.019
[Server] Sent prediction: 3
Received image: min=0.000, max=0.373, mean=0.019
[Server] Sent prediction: 3
Received image: min=0.000, max=0.325, mean=0.019

Received image: min=0.000, max=0.451, mean=0.026
[Server] Sent prediction: 3
Received image: min=0.000, max=0.467, mean=0.026
[Server] Sent prediction: 8
Received image: min=0.000, max=0.533, mean=0.027
[Server] Sent prediction: 3
Received image: min=0.000, max=0.565, mean=0.027
[Server] Sent prediction: 3
Received image: min=0.000, max=0.565, mean=0.026
[Server] Sent prediction: 8
Received image: min=0.000, max=0.451, mean=0.028
[Server] Sent prediction: 3
Received image: min=0.000, max=0.408, mean=0.027
[Server] Sent prediction: 3
Received image: min=0.000, max=0.404, mean=0.027
[Server] Sent prediction: 8
Received image: min=0.000, max=0.431, mean=0.026
[Server] Sent prediction: 5
Received image: min=0.000, max=0.392, mean=0.026
[Server] Sent prediction: 3
Received image: min=0.000, max=0.388, mean=0.027
[Server] Sent prediction: 8
Received image: min=0.000, max=0.392, mean=0.027
[Server] Sent prediction: 3
Received image: min=0.000, max=0.427, mean=0.026
[Server] Sent prediction: 3

Received image: min=0.000, max=0.298, mean=0.020
[Server] Sent prediction: 5
Received image: min=0.000, max=0.314, mean=0.020
[Server] Sent prediction: 5
Received image: min=0.000, max=0.302, mean=0.020
[Server] Sent prediction: 5
Received image: min=0.000, max=0.294, mean=0.020
[Server] Sent prediction: 5
Received image: min=0.000, max=0.369, mean=0.020
[Server] Sent prediction: 5
Received image: min=0.000, max=0.514, mean=0.022
[Server] Sent prediction: 5
Received image: min=0.000, max=0.373, mean=0.022
[Server] Sent prediction: 5
Received image: min=0.000, max=0.333, mean=0.020
[Server] Sent prediction: 5
Received image: min=0.000, max=0.337, mean=0.021
[Server] Sent prediction: 5
Received image: min=0.000, max=0.416, mean=0.023
[Server] Sent prediction: 5
Received image: min=0.000, max=0.443, mean=0.023
[Server] Sent prediction: 5
Received image: min=0.000, max=0.561, mean=0.023
[Server] Sent prediction: 5
Received image: min=0.000, max=0.392, mean=0.021
[Server] Sent prediction: 5

Received image: min=0.000, max=0.357, mean=0.022
[Server] Sent prediction: 5
Received image: min=0.000, max=0.361, mean=0.023
[Server] Sent prediction: 5
Received image: min=0.000, max=0.314, mean=0.022
[Server] Sent prediction: 5
Received image: min=0.000, max=0.314, mean=0.022
[Server] Sent prediction: 8
Received image: min=0.000, max=0.322, mean=0.022
[Server] Sent prediction: 5
Received image: min=0.000, max=0.357, mean=0.022
[Server] Sent prediction: 3
Received image: min=0.000, max=0.380, mean=0.022
[Server] Sent prediction: 8
Received image: min=0.000, max=0.404, mean=0.022
[Server] Sent prediction: 3
Received image: min=0.000, max=0.380, mean=0.022
[Server] Sent prediction: 5
Received image: min=0.000, max=0.384, mean=0.021
[Server] Sent prediction: 5
Received image: min=0.000, max=0.420, mean=0.021
[Server] Sent prediction: 5
Received image: min=0.000, max=0.478, mean=0.022
[Server] Sent prediction: 3
Received image: min=0.000, max=0.486, mean=0.021
[Server] Sent prediction: 8

... (434 outputs left out)

[Server] Sent prediction: 2
Received image: min=0.000, max=0.686, mean=0.018
[Server] Sent prediction: 3
Received image: min=0.000, max=0.627, mean=0.010
[Server] Sent prediction: 3
Received image: min=0.000, max=0.702, mean=0.014
[Server] Sent prediction: 3
Received image: min=0.000, max=0.894, mean=0.024
[Server] Sent prediction: 3
Received image: min=0.000, max=0.914, mean=0.029
[Server] Sent prediction: 3
Received image: min=0.000, max=0.902, mean=0.021
[Server] Sent prediction: 3
Received image: min=0.000, max=0.882, mean=0.022
[Server] Sent prediction: 3
Received image: min=0.000, max=0.875, mean=0.029
[Server] Sent prediction: 7
Received image: min=0.000, max=0.945, mean=0.030
[Server] Sent prediction: 3
Received image: min=0.000, max=0.882, mean=0.029
[Server] Sent prediction: 7
Received image: min=0.000, max=0.671, mean=0.020
[Server] Sent prediction: 1
Received image: min=0.000, max=0.690, mean=0.020
[Server] Sent prediction: 1
Received image: min=0.000, max=0.714, mean=0.021

Received image: min=0.000, max=0.965, mean=0.155
[Server] Sent prediction: 3
Received image: min=0.000, max=0.984, mean=0.146
[Server] Sent prediction: 3
Received image: min=0.000, max=0.961, mean=0.133
[Server] Sent prediction: 3
Received image: min=0.000, max=0.957, mean=0.110
[Server] Sent prediction: 3
Received image: min=0.000, max=0.957, mean=0.088
[Server] Sent prediction: 3
Received image: min=0.000, max=0.988, mean=0.067
[Server] Sent prediction: 3
Received image: min=0.000, max=0.961, mean=0.027
[Server] Sent prediction: 3
Received image: min=0.000, max=0.639, mean=0.012
[Server] Sent prediction: 7
Received image: min=0.000, max=0.749, mean=0.006
[Server] Sent prediction: 7
Received image: min=0.000, max=0.792, mean=0.015
[Server] Sent prediction: 6
Received image: min=0.000, max=0.784, mean=0.023
[Server] Sent prediction: 8
Received image: min=0.000, max=0.788, mean=0.015
[Server] Sent prediction: 8
Received image: min=0.000, max=0.761, mean=0.012
[Server] Sent prediction: 6

Received image: min=0.000, max=0.957, mean=0.156
[Server] Sent prediction: 3
Received image: min=0.000, max=0.992, mean=0.160
[Server] Sent prediction: 3
Received image: min=0.000, max=0.984, mean=0.165
[Server] Sent prediction: 3
Received image: min=0.000, max=0.961, mean=0.173
[Server] Sent prediction: 3
Received image: min=0.000, max=0.980, mean=0.168
[Server] Sent prediction: 3
Received image: min=0.000, max=0.984, mean=0.171
[Server] Sent prediction: 3
Received image: min=0.000, max=0.949, mean=0.168
[Server] Sent prediction: 3
Received image: min=0.000, max=0.984, mean=0.169
[Server] Sent prediction: 3
Received image: min=0.000, max=0.957, mean=0.170
[Server] Sent prediction: 3
Received image: min=0.000, max=0.980, mean=0.173
[Server] Sent prediction: 3
Received image: min=0.000, max=0.969, mean=0.175
[Server] Sent prediction: 3
Received image: min=0.000, max=0.980, mean=0.175
[Server] Sent prediction: 3
Received image: min=0.000, max=0.988, mean=0.177
[Server] Sent prediction: 3

[Server] Sent prediction: 3
Received image: min=0.000, max=0.953, mean=0.120
[Server] Sent prediction: 3
Received image: min=0.000, max=0.949, mean=0.123
[Server] Sent prediction: 3
Received image: min=0.000, max=0.961, mean=0.136
[Server] Sent prediction: 3
Received image: min=0.000, max=0.965, mean=0.146
[Server] Sent prediction: 3
Received image: min=0.000, max=0.992, mean=0.136
[Server] Sent prediction: 3
Received image: min=0.000, max=0.957, mean=0.131
[Server] Sent prediction: 3
Received image: min=0.000, max=0.902, mean=0.123
[Server] Sent prediction: 3
Received image: min=0.000, max=0.961, mean=0.126
[Server] Sent prediction: 3
Received image: min=0.000, max=0.800, mean=0.127
[Server] Sent prediction: 3
Received image: min=0.000, max=0.890, mean=0.138
[Server] Sent prediction: 3
Received image: min=0.000, max=0.875, mean=0.142
[Server] Sent prediction: 3
Received image: min=0.000, max=0.914, mean=0.138
[Server] Sent prediction: 3
Received image: min=0.000, max=0.878, mean=0.138